# 📊 Notebook 05 — VERA Scoring
## VERA: Visual Evidence–Report Alignment

This notebook computes VERA alignment scores for all claims and flags hallucinations.

**Steps:**
1. Load attention maps + extracted claims
2. Compute VERA score for each claim
3. Apply severity-calibrated thresholds
4. Calibrate thresholds on validation split
5. Score test split with calibrated thresholds
6. Save results

**VERA(c) = Σ a(i,j) for (i,j) ∈ R(c) / Σ a(i,j) across entire image**

If VERA(c) < T(c) → flag claim c as likely hallucination

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (auto-detect Kaggle vs local)
if os.path.exists('/kaggle/working'):
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    PROCESSED_DIR, ATTENTION_DIR, CLAIMS_DIR, RESULTS_DIR, FIGURES_DIR,
    SEVERITY_THRESHOLDS, PATCH_GRID_CHEXAGENT, IS_KAGGLE
)
from src.data_utils import load_json, save_json
from src.vera_scorer import (
    score_all_claims, classify_severity, get_threshold,
    calibrate_thresholds, compute_vera_score, compute_attention_entropy,
    SEVERITY_THRESHOLDS as DEFAULT_THRESHOLDS,
)
from src.anatomy_atlas import claim_to_region, get_all_zone_masks

import numpy as np
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter, defaultdict

In [ ]:
# Configuration
USE_MODEL = "chexagent"  # Must match Notebooks 02-03
PATCH_GRID = PATCH_GRID_CHEXAGENT  # (24, 24)

print(f"Model: {USE_MODEL}")
print(f"Patch grid: {PATCH_GRID}")
print(f"\nDefault thresholds:")
for tier, t in DEFAULT_THRESHOLDS.items():
    print(f"  {tier}: {t}")

## 2. Load Data

In [ ]:
# Load inference results
inference_results = load_json(str(ATTENTION_DIR / USE_MODEL / 'inference_results.json'))
inference_results = [r for r in inference_results if 'error' not in r]
print(f"Loaded {len(inference_results)} inference results")

# Create image_id → split mapping
split_map = {}
for split in ['train', 'val', 'test']:
    split_data = load_json(str(PROCESSED_DIR / f'{split}.json'))
    for entry in split_data:
        split_map[entry['image_id']] = split

# Add split info to inference results
for r in inference_results:
    r['split'] = split_map.get(r['image_id'], 'unknown')

# Count per split
split_counts = Counter(r['split'] for r in inference_results)
print(f"\nSplit distribution:")
for split, count in split_counts.items():
    print(f"  {split}: {count}")

## 3. Score All Claims

In [ ]:
print("="*60)
print("Computing VERA Scores")
print("="*60)

all_scored = []  # List of per-image result dicts
all_claims_flat = []  # Flattened list of all scored claims

claims_dir = CLAIMS_DIR / USE_MODEL
attention_dir = ATTENTION_DIR / USE_MODEL

for entry in tqdm(inference_results, desc="VERA scoring"):
    image_id = entry['image_id']
    split = entry.get('split', 'unknown')
    
    # Load claims
    claims_path = claims_dir / f"{image_id}_claims.json"
    if not claims_path.exists():
        continue
    with open(claims_path, 'r') as f:
        claims_data = json.load(f)
    claims = claims_data.get('claims', [])
    
    # Load attention maps
    attn_path = attention_dir / f"{image_id}_attention.npz"
    if not attn_path.exists():
        continue
    attn_data = np.load(str(attn_path))
    attention_maps = attn_data['attention_maps']
    
    # Score all claims
    scored_claims = score_all_claims(
        claims, attention_maps, PATCH_GRID, DEFAULT_THRESHOLDS
    )
    
    # Store result
    result = {
        'image_id': image_id,
        'split': split,
        'generated_report': entry.get('generated_report', ''),
        'claims': scored_claims,
        'num_claims': len(scored_claims),
        'num_flagged': sum(1 for c in scored_claims if c.get('vera_flagged')),
        'num_localizable': sum(1 for c in scored_claims if c.get('localizable')),
    }
    all_scored.append(result)
    
    # Flatten claims with image/split info
    for c in scored_claims:
        c['image_id'] = image_id
        c['split'] = split
    all_claims_flat.extend(scored_claims)

# Summary
total_claims = len(all_claims_flat)
localizable = sum(1 for c in all_claims_flat if c.get('localizable'))
flagged = sum(1 for c in all_claims_flat if c.get('vera_flagged'))
unlocalizable = total_claims - localizable

print(f"\n✅ Scoring complete!")
print(f"   Total claims: {total_claims}")
print(f"   Localizable: {localizable} ({localizable/total_claims*100:.1f}%)")
print(f"   Unlocalizable: {unlocalizable} ({unlocalizable/total_claims*100:.1f}%)")
print(f"   Flagged as hallucination: {flagged} ({flagged/total_claims*100:.1f}%)")

## 4. VERA Score Distribution

In [ ]:
# Get VERA scores by split and flagged status
localizable_claims = [c for c in all_claims_flat if c.get('localizable')]

vera_scores = [c['vera_score'] for c in localizable_claims]
flagged_scores = [c['vera_score'] for c in localizable_claims if c.get('vera_flagged')]
clean_scores = [c['vera_score'] for c in localizable_claims if not c.get('vera_flagged')]

print(f"VERA Score Statistics (localizable claims only):")
print(f"  All:     mean={np.mean(vera_scores):.3f}, std={np.std(vera_scores):.3f}, "
      f"median={np.median(vera_scores):.3f}")
if flagged_scores:
    print(f"  Flagged: mean={np.mean(flagged_scores):.3f}, std={np.std(flagged_scores):.3f}, "
          f"median={np.median(flagged_scores):.3f}")
if clean_scores:
    print(f"  Clean:   mean={np.mean(clean_scores):.3f}, std={np.std(clean_scores):.3f}, "
          f"median={np.median(clean_scores):.3f}")

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# VERA score distribution
if vera_scores:
    axes[0].hist(vera_scores, bins=30, color='#3498db', alpha=0.7, edgecolor='white')
    for tier, thresh in DEFAULT_THRESHOLDS.items():
        axes[0].axvline(x=thresh, linestyle='--', alpha=0.7, label=f'{tier}: T={thresh}')
    axes[0].set_xlabel('VERA Score', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('VERA Score Distribution (All Claims)', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=9)

# Per-severity distribution
severity_data = defaultdict(list)
for c in localizable_claims:
    tier = c.get('severity_tier', 'unknown')
    severity_data[tier].append(c['vera_score'])

tier_colors = {'critical': '#e74c3c', 'moderate': '#f39c12', 'mild': '#2ecc71'}
positions = []
labels = []
data_for_box = []
for tier in ['critical', 'moderate', 'mild']:
    if tier in severity_data:
        data_for_box.append(severity_data[tier])
        labels.append(f"{tier}\n(n={len(severity_data[tier])})")

if data_for_box:
    bp = axes[1].boxplot(data_for_box, labels=labels, patch_artist=True)
    for i, (patch, tier) in enumerate(zip(bp['boxes'], ['critical', 'moderate', 'mild'])):
        patch.set_facecolor(tier_colors.get(tier, '#3498db'))
        patch.set_alpha(0.6)
    axes[1].set_ylabel('VERA Score', fontsize=12)
    axes[1].set_title('VERA Score by Severity Tier', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'vera_score_distribution.png'), dpi=200)
plt.show()

## 5. Threshold Calibration (on Validation Split)

In [ ]:
# For threshold calibration, we need ground truth labels.
# We'll use NLI-based ground truth from the reference reports.
# This is computed here and fully used in Notebook 06.

# Load reference reports for validation split
val_data = load_json(str(PROCESSED_DIR / 'val.json'))
ref_map = {entry['image_id']: entry.get('reference_report', '') for entry in val_data}

# Get validation claims
val_claims = [c for c in all_claims_flat if c.get('split') == 'val' and c.get('localizable')]
print(f"Validation localizable claims: {len(val_claims)}")

# Simple ground truth: check if finding appears in reference report
# (Full NLI-based GT is computed in Notebook 06)
val_gt = []
for claim in val_claims:
    ref_report = ref_map.get(claim.get('image_id', ''), '').lower()
    finding = claim.get('finding', '').lower()
    # Simple heuristic: hallucination if finding not in reference
    is_hallucination = finding not in ref_report if ref_report else True
    val_gt.append(is_hallucination)

print(f"Hallucination rate (simple heuristic): {sum(val_gt)/len(val_gt)*100:.1f}%")

In [ ]:
# Calibrate thresholds
print("\nCalibrating thresholds on validation split...")
val_score_dicts = [{'vera_score': c['vera_score'], 'severity_tier': c.get('severity_tier', 'moderate')}
                   for c in val_claims]

calibrated_thresholds = calibrate_thresholds(
    val_score_dicts, val_gt,
    threshold_range=(0.05, 0.60),
    threshold_step=0.025,
)

print(f"\nCalibrated thresholds:")
for tier, t in calibrated_thresholds.items():
    default = DEFAULT_THRESHOLDS.get(tier, 0.25)
    change = "↑" if t > default else "↓" if t < default else "="
    print(f"  {tier}: {t:.3f} (default was {default}, {change})")

# Save calibrated thresholds
save_json(calibrated_thresholds, str(RESULTS_DIR / 'calibrated_thresholds.json'))

## 6. Re-score with Calibrated Thresholds

In [ ]:
# Re-score all claims with calibrated thresholds
print("Re-scoring all claims with calibrated thresholds...")

for claim in all_claims_flat:
    if not claim.get('localizable') or claim.get('vera_score') is None:
        continue
    
    tier = claim.get('severity_tier', 'moderate')
    new_threshold = calibrated_thresholds.get(tier, DEFAULT_THRESHOLDS.get(tier, 0.25))
    claim['vera_threshold'] = new_threshold
    claim['vera_flagged'] = claim['vera_score'] < new_threshold

# Re-count
new_flagged = sum(1 for c in all_claims_flat if c.get('vera_flagged'))
print(f"\nFlagged with calibrated thresholds: {new_flagged}/{total_claims} "
      f"({new_flagged/total_claims*100:.1f}%)")
print(f"Previous (default thresholds): {flagged}/{total_claims} "
      f"({flagged/total_claims*100:.1f}%)")

## 7. Save Results

In [ ]:
# Save scored results
results_output = RESULTS_DIR / USE_MODEL
results_output.mkdir(parents=True, exist_ok=True)

# Save per-image scored results
# Update the all_scored list with calibrated claims
claims_by_image = defaultdict(list)
for c in all_claims_flat:
    claims_by_image[c.get('image_id', '')].append(c)

for result in all_scored:
    image_id = result['image_id']
    result['claims'] = claims_by_image.get(image_id, [])
    result['num_flagged'] = sum(1 for c in result['claims'] if c.get('vera_flagged'))

save_json(all_scored, str(results_output / 'vera_scores.json'))

# Save flat claims for evaluation
save_json(all_claims_flat, str(results_output / 'vera_claims_flat.json'))

# Save scoring summary
scoring_summary = {
    'model': USE_MODEL,
    'total_claims': total_claims,
    'localizable': localizable,
    'unlocalizable': unlocalizable,
    'flagged_hallucinations': new_flagged,
    'flag_rate': new_flagged / total_claims if total_claims > 0 else 0,
    'default_thresholds': DEFAULT_THRESHOLDS,
    'calibrated_thresholds': calibrated_thresholds,
    'avg_vera_score': float(np.mean(vera_scores)) if vera_scores else 0,
    'avg_vera_flagged': float(np.mean(flagged_scores)) if flagged_scores else 0,
    'avg_vera_clean': float(np.mean(clean_scores)) if clean_scores else 0,
}
save_json(scoring_summary, str(results_output / 'scoring_summary.json'))

print(f"\n📁 Results saved to: {results_output}")
print(f"\nNext: Run 06_evaluation.ipynb for final metrics and paper figures")